
# 2D Ising Model (Two-State Spins) in Pure Python

This is a simple implementation of the **2D, two-state Ising model** on an $L \times L$ square lattice with **periodic boundary conditions**, using **Metropolis Monte Carlo** updates. The model computes the magnetization of a system at a given temperature $T$.


## Model definition

### Spins and lattice
We have a square grid of spins
$$
s_{i,j} \in \{-1, +1\}, \quad i,j \in \{0,\dots,L-1\}.
$$

### Hamiltonian (energy)

The energy of the system is defined by
$$
E(\{s\}) \;=\; -J \sum_{\langle (i,j),(k,\ell)\rangle} s_{i,j}\,s_{k,\ell}\;-\;h\sum_{i,j} s_{i,j}.
$$

where $J$ is a measure of how easy it is for two neighbors to interact (it's called the coupling constant) and $h$ is the strength of some external field. For simplicity, we set $h=0$ as to not have to worry about external fields (such as a magnetic or electric field).


- The notation $\langle \cdot,\cdot \rangle$ means **nearest neighbors** (up, down, left, right).
- For edge sites, we apply periodic boundary conditions that “wrap around,” so every site has exactly 4 neighbors. For example, site $s_{00}$ still has four nearest neighbors: directly to the right ($s_{0,1}$) and below ($s_{10,}$) but also above it at ($s_{L-1, 0}$) and to its left ($s_{0,L-1}$). The figure below illustrate the periodic boundary conditions for a lattice with $L=6$.

<center>

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/numerical_algorithms/periodic_boundary_ising.png)

</center>


### Magnetization
The total magnetization is:
$$
M \;=\;\sum_{i,j} s_{i,j},
$$
and magnetization per spin is:
$$
m \;=\;\frac{M}{L^2}.
$$


## Metropolis Monte Carlo dynamics

A **single-spin flip proposal** chooses a random lattice site $(i,j)$ and proposes:
$$
s_{i,j} \leftarrow -s_{i,j}.
$$

### Energy change for a proposed flip
Let the sum of the four nearest neighbors be:
$$
S_{\text{nn}}(i,j) = s_{i-1,j} + s_{i+1,j} + s_{i,j-1} + s_{i,j+1},
$$
(with indices understood modulo $L$ for periodic boundaries).

If we flip $s_{i,j}$, the energy change is:
$$
\Delta E \;=\; 2\,s_{i,j}\,\big(J\,S_{\text{nn}}(i,j) + h\big).
$$

This formula is key: it lets us do local updates without recomputing the full energy.

### Acceptance rule for the flip proposal
At temperature $T>0$, the Metropolis acceptance probability for the flip $s_{i,j} \leftarrow -s_{i,j}$ is:
$$
P(\text{accept}) =
\begin{cases}
1, & \Delta E \le 0, \\
e^{-\Delta E/T}, & \Delta E > 0.
\end{cases}
$$

A **sweep** (or Monte Carlo step per spin) is typically defined as $L^2$ attempted flips.

---

## Pure-Python implementation

- Uses lists-of-lists for spins.
- Periodic boundaries via modulo indexing.
- Tracks magnetization incrementally.
- Provides a full energy function (useful for diagnostics).

---

## How to use / what to expect

### Typical workflow

1. Create a model with chosen $(L, T, J, h)$.
2. Run a number of sweeps to **thermalize** (burn-in).
3. Then measure observables (energy, magnetization) every sweep or every few sweeps.

### Notes on temperature

* At low $T$, spins tend to align (large $|m|$).
* At high $T$, spins look disordered ($m \approx 0$).
* For the 2D Ising model at $h=0$ and $J=1$, the critical temperature is approximately
  $T_c \approx 2.269.$


In [5]:
from __future__ import annotations
import math
import random


class Ising2D:
    """
    Simple 2D Ising model with periodic boundary conditions.
    Spins s[i][j] ∈ {+1, -1}.
    Energy: E = -J * sum_<ij> s_i s_j - h * sum_i s_i
    (Nearest-neighbor, each bond counted once.)
    """

    def __init__(
        self,
        L: int,
        T: float,
        J: float = 1.0,
        h: float = 0.0,
        seed: int | None = None,
        init: str = "random",  # "random", "up", "down"
    ) -> None:
        if L <= 1:
            raise ValueError("L must be > 1.")
        if T <= 0:
            raise ValueError("T must be > 0.")
        self.L = L
        self.T = T
        self.J = J
        self.h = h
        self.rng = random.Random(seed)

        # spins as list-of-lists of ints (+1 / -1)
        if init == "up":
            self.s = [[1 for _ in range(L)] for _ in range(L)]
        elif init == "down":
            self.s = [[-1 for _ in range(L)] for _ in range(L)]
        elif init == "random":
            self.s = [[1 if self.rng.random() < 0.5 else -1 for _ in range(L)] for _ in range(L)]
        else:
            raise ValueError("init must be one of: 'random', 'up', 'down'.")

        # Track magnetization for convenience (optional but cheap)
        self.M = sum(sum(row) for row in self.s)

    def _nn_sum(self, i: int, j: int) -> int:
        """Sum of 4 nearest neighbors with periodic boundary conditions."""
        L = self.L
        s = self.s
        return (
            s[(i - 1) % L][j]
            + s[(i + 1) % L][j]
            + s[i][(j - 1) % L]
            + s[i][(j + 1) % L]
        )

    def delta_E_if_flip(self, i: int, j: int) -> float:
        """
        Energy change if we flip s_ij -> -s_ij.
        ΔE = 2 * s_ij * (J * sum_nn + h)
        """
        sij = self.s[i][j]
        nn = self._nn_sum(i, j)
        return 2.0 * sij * (self.J * nn + self.h)

    def metropolis_step(self) -> bool:
        """
        Attempt one random single-spin flip (Metropolis).
        Returns True if accepted, False otherwise.
        """
        i = self.rng.randrange(self.L)
        j = self.rng.randrange(self.L)

        dE = self.delta_E_if_flip(i, j)
        if dE <= 0.0:
            # accept downhill moves
            self._flip(i, j)
            return True

        # accept uphill with probability exp(-ΔE/T)
        if self.rng.random() < math.exp(-dE / self.T):
            self._flip(i, j)
            return True
        return False

    def sweep(self) -> float:
        """
        One Monte Carlo sweep: L*L Metropolis attempts.
        Returns acceptance ratio.
        """
        n = self.L * self.L
        acc = 0
        for _ in range(n):
            if self.metropolis_step():
                acc += 1
        return acc / n

    def _flip(self, i: int, j: int) -> None:
        """Flip spin and update magnetization."""
        old = self.s[i][j]
        self.s[i][j] = -old
        self.M += -2 * old  # since new - old = (-old) - old = -2old

    def magnetization_per_spin(self) -> float:
        return self.M / (self.L * self.L)

    def energy(self) -> float:
        """
        Compute total energy E (bonds counted once: right + down).
        """
        L = self.L
        s = self.s
        J = self.J
        h = self.h

        e_bonds = 0
        m = 0
        for i in range(L):
            for j in range(L):
                sij = s[i][j]
                m += sij
                # count each bond once: to the right and down
                e_bonds += sij * (s[i][(j + 1) % L] + s[(i + 1) % L][j])

        return -J * e_bonds - h * m

    def ascii_snapshot(self, up: str = "🟩 ", down: str = "🟦 ") -> str:
        """Quick-and-dirty visualization."""
        lines = []
        for row in self.s:
            lines.append("".join(up if v == 1 else down for v in row))
        return "\n".join(lines)


def run_demo() -> None:
    L = 16
    # Study a range of temperatures, including near the critical point 
    # (T_c ≈ 2.269 for J=1, h=0).
    temperatures = [0.0001, 1.5, 2.269, 6] 
    for T in temperatures:
        print(f"\n=== T={T} ===")
        model = Ising2D(L=L, T=T, J=1.0, h=0.0, seed=0, init="random")

        # Burn-in: run some sweeps to reach equilibrium before sampling.
        for _ in range(1000):
            model.sweep()

        # Sample
        for k in range(10):
            acc = model.sweep()
            #print(f"sweep {k:02d}  acc={acc:.3f}  E={model.energy():.1f}  m={model.magnetization_per_spin():+.3f}")

        print("\nSnapshot:\n")
        print(model.ascii_snapshot())


if __name__ == "__main__":
    run_demo()


=== T=0.0001 ===

Snapshot:

🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 
🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 🟩 

=== T=1.5 ===

Snapshot:

🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 
🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦 🟦